# Who owns this alarm?

An automated vehicle is in service. A monitor output goes abnormal:
detections drop in one region of the scene. Before anyone can act on
it, the anomaly has to belong to a concern. Is this a performance
insufficiency, which ISO 21448 owns? An attack, which ISO/SAE 21434
owns? A hardware fault, which ISO 26262 owns?

This notebook follows that single anomaly clause by clause and asks
each standard whether it owns it. The walk ends without an answer.
That absence is the subject of the position paper, and this notebook
is the paper's argument made executable rather than a new claim.

You do not have to run anything. Each conclusion is stated before the
cell that shows it. Running the cells lets you change the anomaly and
watch the same dead end appear.

In [ ]:
# On Colab, fetch the repository. Locally, this does nothing.
try:
    import src.analysis.anomaly_walk  # noqa: F401
except ImportError:  # pragma: no cover - Colab only
    import subprocess, sys, os
    subprocess.run(["git", "clone", "--quiet", "https://github.com/milinpatel07/Assurance-Gaps-in-an-Integrated-Safety-and-Cybersecurity-Case.git", "repo"], check=True)
    os.chdir("repo")
    sys.path.insert(0, os.getcwd())

from src.analysis.anomaly_walk import (
    SCENARIOS,
    AnomalyObservation,
    Cause,
    resolve_with_cause,
    walk_result,
)

## 1. What the monitor sees

A runtime monitor reports what it can measure. It reports that
detections dropped and that the ensemble members disagree. It reports
that the hardware diagnostic passed, so nothing indicates a random
hardware fault.

There is no field for the cause. That is not an omission in this
notebook. It is the situation: a perception function has no ground
truth to check its own output against while the vehicle is driving.

In [ ]:
anomaly = SCENARIOS["detections_drop"]
print(anomaly.description)
for line in anomaly.observable_summary():
    print("  ", line)

## 2. Ask each standard whether it owns the anomaly

Each standard answers from its own clauses. Read the reasons rather
than the verdicts: every one of them is conditional on something the
monitor did not report.

In [ ]:
result = walk_result(anomaly)

for v in result["verdicts"]:
    print(f"{v.standard_id} {v.clause} ({v.clause_title})")
    print(f"  verdict: {v.verdict.value}")
    print(f"  because: {v.reason}")
    print()

## 3. The walk ends without an assignment

ISO 21448 owns the anomaly if the cause is a performance
insufficiency, and its Clause 1 refers the attack case away to
ISO/SAE 21434. ISO/SAE 21434 owns the anomaly if the cause is an
attack, and it does not address performance insufficiency. Each
waits on the same fact, and no clause establishes it.

So the anomaly is not assigned. Not assigned is different from
assigned to nobody: every standard is behaving correctly inside its
own scope. The gap is between the scopes.

In [ ]:
print("assigned to:", result["assigned_to"])
print("conditional on a cause nobody establishes:")
for v in result["conditional_on_cause"]:
    print(f"  {v.standard_id} {v.clause}")

## 4. Does a weather report settle it?

The obvious objection: if it was raining, the cause is weather, so
ISO 21448 owns it. Try it. Try a reported security event too, and
both at once.

The assignment does not change. A correlated report is not the cause.
The position paper makes this point directly: the same missing region
follows from rain, sparse returns or a rare object pose, which is a
performance insufficiency, and from a spoofing or relay attack, which
is a cybersecurity event. The observation is identical either way, so
a report that something else happened at the same time does not
establish what caused this.

In [ ]:
for name, obs in SCENARIOS.items():
    r = walk_result(obs)
    print(f"{name:26} assigned to: {r['assigned_to']}")

## 5. The one case that does resolve

The failing hardware diagnostic is the exception, and it is the
exception for a reason worth noticing. ISO 26262 requires diagnostic
mechanisms that identify random hardware faults, so in that one case
a clause does establish the cause, and the assignment follows.

That is what the other three concerns lack: not a rule for deciding
ownership once the cause is known, but any mechanism that establishes
the cause from what the vehicle can observe.

In [ ]:
hw = SCENARIOS["hardware_fault"]
print(hw.description)
print("assigned to:", walk_result(hw)["assigned_to"])

## 6. What would settle it

If the cause were known, the assignment is immediate and
uncontroversial. The function below is not a proposal and not a
method. It takes the cause as an input and shows that once you have
it, nothing else is missing.

This isolates what the gap actually is. It is not that the standards
disagree about who should own which cause. They agree. It is that
nothing turns an observation into a cause while the vehicle is in
service, and the position paper calls that the missing assignment
step.

In [ ]:
for cause in Cause:
    owner = resolve_with_cause(anomaly, cause)
    print(f"if the cause were {cause.value:28} -> {owner}")

## 7. Change the anomaly yourself

Build any observation from what a monitor can report and run the walk
on it. The unassigned result is not a property of the example chosen
here; it follows from the clause structure, so it survives any
observation that does not indicate a hardware fault.

In [ ]:
mine = AnomalyObservation(
    "My own anomaly",
    detections_dropped=True,
    ensemble_disagreement_high=False,
    hardware_diagnostic_passed=True,
    security_event_reported=True,
    adverse_weather_reported=True,
)
print("assigned to:", walk_result(mine)["assigned_to"])

## Where this goes next

Assignment is the first of two missing steps. Suppose the anomaly had
been assigned, and every concern re-evaluated its own claim. Those
results still have to support one top claim, on measurement scales
that do not convert into one another, and no clause combines them
either.

That second step is the same problem the other paper reports before
release, at the goal where all four standards meet. The repository
sets the two side by side on its seam page.

- The argument, one node at a time: `docs/gsn_view.html`
- One problem at two lifecycle points: `docs/seam.html`
- Where every claim comes from: `TRACEABILITY.md`

The clause-level reasons in this notebook come from
`src/analysis/anomaly_walk.py`, which cites the clause behind every
verdict, and `tests/test_anomaly_walk.py` checks them.